# The Facebook Hearing, Step by Step

| Item | Description |
|---|---|
| Goal | Follow one hearing through the whole of T1 using the saved production run: *Facebook, Social Media Privacy, and the Use and Abuse of Data*, the joint Senate Judiciary and Commerce hearing of April 10, 2018 with Mark Zuckerberg, GovInfo package `CHRG-115shrg37801`. The transcript becomes 903 speaking turns, 55 chunks, 55 segment summaries, 3 condensed accounts, and one 264-word summary, which is then set beside C-SPAN's abstract. Every model input is rebuilt and checked against the hash the run logged, so the chain is verified without calling the model. |
| Input | The transcript from D1.1; the hearing's turns from `hearings_sst.parquet` of D1.4; its 59 logged calls, its summary, and its C-SPAN match from T1.2 and T1.3; the production tokenizer from the project cache. |
| Output | Readable copies beside this notebook, kept in the repository because the proposal links to them: `01 - Inputs/source.txt`, `01 - Inputs/parsed_turns.json`, `01 - Inputs/prompts.md` with the three prompts, `01 - Inputs/Chunks/chunks.md` with six pages of chunks, `02 - Outputs/outputs.md` with every recorded output and the abstract, and `03 - Logs and Verification/call_log.jsonl`, a copy of the production log. |
| Run | Executed on 2026-10-31. No model is called; the tokenizer is the only model file read. The copies it writes are identical to the ones exported in August. |

## Step 1. Import packages

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from transformers import AutoTokenizer

## Step 2. Define file locations

In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T1 - Text Summarization/T1.4 - Running Example [Facebook-Cambridge Analytica Scandal Hearing]")
NOTEBOOK_NAME = "T1.4.1 - Example - The Facebook Hearing Step by Step"
TRANSCRIPT_FOLDER = SCRATCH_FOLDER / "D1 - Data Collection and Processing/D1.1 - GovInfo Congressional Hearings" / "Raw"
SAMPLE_FOLDER = SCRATCH_FOLDER / "D1 - Data Collection and Processing/D1.4 - SST Sample Construction" / "Processed"
RUN_FOLDER = SCRATCH_FOLDER / "T1 - Text Summarization/T1.2 - Process Entire Sample Congressional Hearings" / "Interim" / "sst_Qwen3.5-4B"
SUMMARIES_FOLDER = SCRATCH_FOLDER / "T1 - Text Summarization/T1.2 - Process Entire Sample Congressional Hearings" / "Processed"
MATCHES_FOLDER = SCRATCH_FOLDER / "T1 - Text Summarization/T1.3 - Validation of Summaries" / "Processed"
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
EXAMPLE_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER  # the readable copies live beside this notebook
INPUTS_FOLDER = EXAMPLE_FOLDER / "01 - Inputs"
CHUNKS_FOLDER = INPUTS_FOLDER / "Chunks"
OUTPUTS_FOLDER = EXAMPLE_FOLDER / "02 - Outputs"
LOGS_FOLDER = EXAMPLE_FOLDER / "03 - Logs and Verification"

import os
os.environ["HF_HOME"] = str(MODEL_CACHE)
for folder in (CHUNKS_FOLDER, OUTPUTS_FOLDER, LOGS_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)

## Step 3. Load the hearing, its run, its summary, and its C-SPAN match

The call log holds every model call the production run made for this hearing, in order. Each line carries a hash of the input and the full output.

In [3]:
PACKAGE = "CHRG-115shrg37801"
MODEL = "Qwen/Qwen3.5-4B"
CHUNK_TOKENS = 6144


def one_row(path, column="packageId"):
    rows = pq.read_table(path, filters=[(column, "==", PACKAGE)]).to_pylist()
    assert len(rows) == 1, f"{len(rows)} rows for {PACKAGE} in {path.name}"
    return rows[0]


hearing = one_row(SAMPLE_FOLDER / "hearings_sst.parquet")
summary = one_row(SUMMARIES_FOLDER / "hearing_summaries.parquet")
reference = one_row(MATCHES_FOLDER / "cspan_matches.parquet")
calls = [json.loads(line) for line in (RUN_FOLDER / "status" / f"{PACKAGE}.jsonl").read_text().splitlines() if line.strip()]
turns = hearing["turns"]

print(hearing["title"])
print(f"{hearing['chamber']}, held {hearing['heldDate_first']}, committees {hearing['committee_ids']}")
print(f"{hearing['n_words']:,} words in {len(turns):,} turns; {hearing['n_paragraphs']:,} paragraphs, {hearing['n_paragraphs_dropped']} dot-leader paragraphs dropped")
print(f"production run: {summary['n_chunks']} chunks, {summary['n_calls']} calls, {summary['n_collapse_rounds']} collapse round, {summary['summary_tokens']} summary tokens")
stages = pd.Series([c["stage"] for c in calls]).value_counts().to_dict()
print(f"logged calls: {len(calls)} {stages}; model {calls[0]['model']}, prompt version {calls[0]['prompt_version']}")
print(f"C-SPAN match: program {reference['cspan_id']}, score {reference['score']:.2f}: {reference['cspan_title']}")

FACEBOOK, SOCIAL MEDIA PRIVACY, AND THE USE AND ABUSE OF DATA
SENATE, held 2018-04-10, committees ['sscm00', 'ssju00']
243,434 words in 903 turns; 8,192 paragraphs, 40 dot-leader paragraphs dropped
production run: 55 chunks, 59 calls, 1 collapse round, 359 summary tokens
logged calls: 59 {'map': 55, 'collapse': 3, 'reduce': 1}; model Qwen/Qwen3.5-4B, prompt version hearing-mapreduce-1.0
C-SPAN match: program 500690, score 0.73: Facebook CEO Mark Zuckerberg Hearing on Data Privacy and Protection


## Step 4. Read the transcript as speaking turns

D1.1.4 split the transcript into paragraphs on the four-space indent and grouped them into turns that open with a title and a surname. The second turn opens with Chairman Grassley, and Mr. Zuckerberg's turns carry no `bioGuideId` because he is not on the roster. The saved parse found no written-question block, so the appendix of written answers stays inside the last turn, 159,475 words under Grassley's name. The example keeps that limitation of the run.

In [4]:
source_text = BeautifulSoup((TRANSCRIPT_FOLDER / str(hearing["year"]) / "html" / f"{PACKAGE}.htm").read_bytes(), "html.parser").get_text()
print(f"source text: {len(source_text):,} characters, {len(source_text.split()):,} words\n")

for i, turn in enumerate(turns[:4]):
    print(f"turn {i + 1}: {turn['honorific']} {turn['surname']} | bioGuideId {turn['bioGuideId']} | member {turn['is_member']} | section {turn['section']} | {turn['n_words']:,} words")
    print(f"   {turn['text'][:160]!r}\n")
longest = max(turns, key=lambda t: t["n_words"])
print(f"longest turn: number {turns.index(longest) + 1}, {longest['honorific']} {longest['surname']}, {longest['n_words']:,} words")
print(f"turns by section: {pd.Series([t['section'] for t in turns]).value_counts().to_dict()}")

source text: 1,638,635 characters, 243,946 words

turn 1: None None | bioGuideId None | member False | section spoken | 645 words
   '- FACEBOOK, SOCIAL MEDIA PRIVACY, AND THE USE AND ABUSE OF DATA [Senate Hearing 115-683] [From the U.S. Government Publishing Office]\n\nS. Hrg. 115-683\n\nFACEBOOK'

turn 2: Chairman Grassley | bioGuideId G000386 | member True | section spoken | 285 words
   "Chairman Grassley. The committees on the Judiciary and Commerce, Science, and Transportation will come to order.\n\nWe welcome everyone to today's hearing on ``Fa"

turn 3: Chairman Thune | bioGuideId T000250 | member True | section spoken | 775 words
   "Chairman Thune. Thank you, Chairman Grassley.\n\nToday's hearing is extraordinary. It is extraordinary to hold a joint committee hearing. It is even more extraord"

turn 4: Chairman Grassley | bioGuideId G000386 | member True | section spoken | 18 words
   'Chairman Grassley. Thank you. And now, Ranking Member Feinstein.\n\nSTATEMENT OF HON. DIANN

## Step 5. Define the three prompts

The prompts of the production run, under the version stamped on every logged call.

In [5]:
# One prompt for all three stages fails in a specific way: the reduce step then treats summaries as speech and re-attributes positions.
PROMPT_VERSION = "hearing-mapreduce-1.0"  # stamped on every logged call, so a summary can be traced to the wording that produced it
REDUCE_MAX_TOKENS = 512  # the summaries are later embedded by a 512-token encoder, so a longer summary would be cut, not longer
REDUCE_TARGET_WORDS = "200 to 250"  # the model writes about 1.55 tokens per word, so this lands near 390 tokens with room to finish
GENERATION = {"do_sample": True, "temperature": 0.1}  # near-greedy sampling, kept from the earlier pipeline

MAP_SYSTEM = ("You are an expert in text summarization with a working knowledge of United "
              "States congressional hearings. You report only what the transcript states.")

MAP_USER = """Below is one consecutive segment of a congressional hearing transcript. Each speaking turn is labelled with the speaker.

Summarize this segment in about 150 words. Cover:
- the subjects raised, and what was said about them;
- the position each named speaker takes, attributed to that speaker;
- any numbers, dollar amounts, dates, and bill or act names, copied exactly as they appear.

If the segment is procedural or ceremonial and contains none of these, say so in one sentence.

--- SEGMENT ---
{text}
--- END OF SEGMENT ---

Write only the summary of the segment above."""

COLLAPSE_SYSTEM = ("You are an expert in text summarization with a working knowledge of United "
                   "States congressional hearings. You condense summaries without adding to them.")

COLLAPSE_USER = """Below are summaries of consecutive segments of one congressional hearing, in the order they occurred. They are already summaries, not transcript.

Condense them into one account of about 200 words that keeps the named speakers, their positions, and the numbers and legislation. This is a middle portion of the hearing, so do not write an overall conclusion about the hearing.

--- SEGMENT SUMMARIES ---
{text}
--- END ---

Write only the condensed account."""

REDUCE_SYSTEM = ("You are an expert in text summarization with a working knowledge of United "
                 "States congressional hearings. You write from the summaries given and add "
                 "nothing that is not in them.")

REDUCE_USER = """Below are summaries of every segment of one congressional hearing, in order. They are already summaries, not transcript.

Write a single summary of the hearing as a whole, in about {target_words} words. Cover:
- what the hearing was about;
- the main subjects, the most heavily discussed first;
- the position each named speaker took;
- any point where speakers took opposing positions, naming both sides;
- the key numbers and the legislation referred to, exactly as given.

Where the segment summaries do not establish one of these, leave it out rather than inferring it.

--- SEGMENT SUMMARIES ---
{text}
--- END ---

Write only the summary of the hearing, in about {target_words} words, as continuous prose."""

## Step 6. Define how speaking turns are packed into chunks

The same packing as T1.2.1: whole turns in order within the budget, a turn longer than the budget divided at paragraph boundaries.

In [6]:
def format_turn(turn):
    """One turn as the model sees it: a speaker label, then what was said. The chair has no surname, so the honorific alone is the label."""
    name = turn.get("surname") or ""
    honorific = turn.get("honorific") or ""
    who = f"{honorific} {name}".strip() or "Unidentified speaker"
    if turn.get("section") == "qfr":
        who = f"{who} (submitted for the record)"
    return f"{who}: {turn['text']}"


def encoded_length(tokenizer, text):
    return len(tokenizer.encode(text, add_special_tokens=False))


def token_slice(text, tokenizer, max_tokens):
    """Cut text into pieces that each re-encode within the budget. decode then encode is not the identity, so every piece is re-checked."""
    ids = tokenizer.encode(text, add_special_tokens=False)
    pieces = []
    start = 0
    while start < len(ids):
        take = min(max_tokens, len(ids) - start)
        piece = tokenizer.decode(ids[start:start + take])
        while take > 64 and encoded_length(tokenizer, piece) > max_tokens:
            take = int(take * 0.9)
            piece = tokenizer.decode(ids[start:start + take])
        pieces.append(piece)
        start += take
    return pieces


def split_long_turn(text, tokenizer, max_tokens):
    """Divide one over-long turn at paragraph boundaries, and a paragraph that is still too long by tokens."""
    join_cost = encoded_length(tokenizer, "\n\n")
    out = []
    buffer = []
    buffer_tokens = 0
    for paragraph in text.split("\n\n"):
        n = encoded_length(tokenizer, paragraph)
        if n > max_tokens:
            if buffer:
                out.append("\n\n".join(buffer))
                buffer, buffer_tokens = [], 0
            out.extend(token_slice(paragraph, tokenizer, max_tokens))
            continue
        cost = n + (join_cost if buffer else 0)
        if buffer_tokens + cost > max_tokens:
            out.append("\n\n".join(buffer))
            buffer, buffer_tokens = [], 0
            cost = n
        buffer.append(paragraph)
        buffer_tokens += cost
    if buffer:
        out.append("\n\n".join(buffer))
    # Re-encode every piece: separators cost tokens and the tokenizer re-merges across a cut.
    verified = []
    for piece in out:
        if encoded_length(tokenizer, piece) <= max_tokens:
            verified.append(piece)
        else:
            verified.extend(token_slice(piece, tokenizer, max_tokens))
    return verified


def chunk_turns(turns, tokenizer, max_tokens, spoken_only=True):
    """Group consecutive whole turns into chunks of at most max_tokens tokens. Only a turn longer than the budget is divided."""
    selected = [t for t in turns if not (spoken_only and t.get("section") == "qfr")]
    join_cost = encoded_length(tokenizer, "\n\n")  # the blank lines between turns are tokens too
    chunks = []
    current = []
    current_tokens = 0
    current_speakers = []

    def flush(split=False):
        nonlocal current, current_tokens, current_speakers
        if not current:
            return
        text = "\n\n".join(current)
        speakers = sorted(set(current_speakers))
        if encoded_length(tokenizer, text) > max_tokens:  # the emitted text is what counts, not the sum of its parts
            for piece in split_long_turn(text, tokenizer, max_tokens):
                chunks.append({"text": piece, "n_turns": len(current), "n_tokens": encoded_length(tokenizer, piece),
                               "speakers": speakers, "split_turn": True})
        else:
            chunks.append({"text": text, "n_turns": len(current), "n_tokens": encoded_length(tokenizer, text),
                           "speakers": speakers, "split_turn": split})
        current, current_tokens, current_speakers = [], 0, []

    for turn in selected:
        rendered = format_turn(turn)
        n = len(tokenizer.encode(rendered, add_special_tokens=False))
        if n > max_tokens:  # one turn that cannot fit: emit what is pending, then divide this turn
            flush()
            for piece in split_long_turn(rendered, tokenizer, max_tokens):
                chunks.append({"text": piece, "n_turns": 1, "n_tokens": encoded_length(tokenizer, piece),
                               "speakers": [turn.get("surname") or "Unidentified speaker"], "split_turn": True})
            continue
        cost = n + (join_cost if current else 0)
        if current_tokens + cost > max_tokens:
            flush()
            cost = n
        current.append(rendered)
        current_tokens += cost
        current_speakers.append(turn.get("surname") or "Unidentified speaker")
    flush()
    return chunks

## Step 7. Rebuild the 55 chunks with the production tokenizer

The tokenizer is the one the run used, read from the project cache. The count and every chunk's size are checked against the budget and the run.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
chunks = chunk_turns(turns, tokenizer, CHUNK_TOKENS, spoken_only=True)

assert len(chunks) == summary["n_chunks"] == 55, f"{len(chunks)} chunks rebuilt, the run had {summary['n_chunks']}"
assert all(c["n_tokens"] <= CHUNK_TOKENS for c in chunks), "a chunk exceeds the budget"
sizes = [c["n_tokens"] for c in chunks]
print(f"{len(chunks)} chunks; tokens per chunk min {min(sizes):,}, median {sorted(sizes)[len(sizes) // 2]:,}, max {max(sizes):,}")
print(f"chunks from a split turn: {sum(c['split_turn'] for c in chunks)}")
print(f"turns per chunk: {[c['n_turns'] for c in chunks]}")

55 chunks; tokens per chunk min 1,937, median 6,103, max 6,144
chunks from a split turn: 41
turns per chunk: [9, 9, 72, 118, 49, 53, 46, 1, 1, 1, 1, 1, 1, 1, 99, 67, 81, 64, 107, 96, 31, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## Step 8. Verify every logged call

Each map prompt is rebuilt from its chunk and its hash compared with the log. The 55 segment summaries from the log are then grouped within the budget exactly as the run grouped them. Each collapse prompt is rebuilt and compared, and finally the reduce prompt. The reduce output in the log must equal the stored summary.

In [8]:
def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def encoded_length(text):
    return len(tokenizer.encode(text, add_special_tokens=False))


map_calls = [c for c in calls if c["stage"] == "map"]
assert len(map_calls) == len(chunks), f"{len(map_calls)} map calls for {len(chunks)} chunks"
for i, (chunk, call) in enumerate(zip(chunks, map_calls)):
    assert call["index"] == i and sha256(MAP_USER.format(text=chunk["text"])) == call["input_sha256"], f"map prompt {i + 1} does not match the log"
print(f"{len(map_calls)} map prompts rebuilt and matched to the log")

pending = [c for c in calls if c["stage"] != "map"]
level_summaries = [c["output"] for c in map_calls]
level = 0
while encoded_length("\n\n".join(level_summaries)) > CHUNK_TOKENS:
    level += 1
    groups, current, current_tokens = [], [], 0
    for text in level_summaries:
        n = encoded_length(text)
        if current and current_tokens + n > CHUNK_TOKENS:
            groups.append(current)
            current, current_tokens = [], 0
        current.append(text)
        current_tokens += n
    if current:
        groups.append(current)
    collapsed = []
    for i, group in enumerate(groups):
        call = pending.pop(0)
        assert (call["stage"], call["level"], call["index"], call["n_inputs"]) == ("collapse", level, i, len(group)), f"collapse {level}.{i} out of order"
        assert sha256(COLLAPSE_USER.format(text="\n\n".join(group))) == call["input_sha256"], f"collapse prompt {level}.{i} does not match the log"
        collapsed.append(call["output"])
    print(f"collapse round {level}: {len(groups)} groups of {[len(g) for g in groups]} summaries, prompts matched to the log")
    level_summaries = collapsed

reduce_call = pending.pop(0)
assert reduce_call["stage"] == "reduce" and not pending, "unexpected calls left in the log"
assert sha256(REDUCE_USER.format(text="\n\n".join(level_summaries), target_words=REDUCE_TARGET_WORDS)) == reduce_call["input_sha256"], "the reduce prompt does not match the log"
assert reduce_call["output"] == summary["summary"], "the logged reduce output differs from the stored summary"
print(f"reduce prompt matched; its output is the stored summary: {len(summary['summary'].split())} words, {summary['summary_tokens']} tokens")
print(f"\nall {len(calls)} calls verified without calling the model")

55 map prompts rebuilt and matched to the log
collapse round 1: 3 groups of [23, 22, 10] summaries, prompts matched to the log
reduce prompt matched; its output is the stored summary: 264 words, 359 tokens

all 59 calls verified without calling the model


## Step 9. Read the chain for one chunk

Chunk 2 holds Mr. Zuckerberg's opening statement and Chairman Grassley's first questions. Below it come its segment summary and the first condensed account, which covers segment summaries 1 to 23. Then the final summary and C-SPAN's abstract of the same hearing.

In [9]:
collapse_calls = [c for c in calls if c["stage"] == "collapse"]
display(Markdown(f"**Chunk 2, first 1,500 of {len(chunks[1]['text']):,} characters** ({chunks[1]['n_tokens']:,} tokens, speakers {', '.join(chunks[1]['speakers'])})\n\n```text\n{chunks[1]['text'][:1500]}\n```"))
display(Markdown(f"**Segment summary of chunk 2** ({map_calls[1]['output_tokens']} tokens)\n\n```text\n{map_calls[1]['output']}\n```"))
display(Markdown(f"**Condensed account 1 of {len(collapse_calls)}**, from segment summaries 1 to {collapse_calls[0]['n_inputs']} ({collapse_calls[0]['output_tokens']} tokens)\n\n```text\n{collapse_calls[0]['output']}\n```"))
display(Markdown(f"**Final summary** ({summary['summary_tokens']} tokens)\n\n```text\n{summary['summary']}\n```"))
display(Markdown(f"**C-SPAN abstract**, program {reference['cspan_id']}: {reference['cspan_title']}\n\n```text\n{reference['abstract']}\n```"))

**Chunk 2, first 1,500 of 28,085 characters** (5,898 tokens, speakers Grassley, Zuckerberg)

```text
Mr. Zuckerberg: Mr. Zuckerberg. Chairman Grassley, Chairman Thune, Ranking Member Feinstein, and Ranking Member Nelson and members of the Committee, we face a number of important issues around privacy, safety, and democracy, and you will rightfully have some hard questions for me to answer. Before I talk about the steps we are taking to address them, I want to talk about how we got here.

Facebook is an idealistic and optimistic company. For most of our existence, we focused on all of the good that connecting people can do. And as Facebook has grown, people everywhere have gotten a powerful new tool for staying connected to the people they love, for making their voices heard, and for building communities and businesses. Just recently, we have seen the #metoo movement and the March for Our Lives organized, at least in part, on Facebook. After Hurricane Harvey, people came together to raise more than $20 million for relief. And more than 70 million small businesses use Facebook to create jobs and grow.

But it is clear now that we did not do enough to prevent these tools for being used for harm as well, and that goes for fake news, for foreign interference in elections, and hate speech, as well as developers and data privacy. We did not take a broad enough view of our responsibility, and that was a big mistake. And it was my mistake, and I am sorry. I started Facebook, I run it, and I am responsible for what happens here.

So now, we have to go through all of our relationship w
```

**Segment summary of chunk 2** (273 tokens)

```text
Mark Zuckerberg addressed the Committee on privacy, safety, and democracy, acknowledging a failure to prevent harm from fake news, foreign interference, and hate speech. He apologized for the Cambridge Analytica scandal, where Aleksandr Kogan's app accessed data from tens of millions of users. Zuckerberg stated that while Facebook made changes in 2014 to restrict app data access, they must now investigate tens of thousands of apps, having already suspended around 200 apps pending audit. He confirmed that since 2006, Facebook sent over 1,150 cease-and-desist letters to over 1,600 targets and took action against about 370,000 apps in 2017. Regarding Russian election interference, Zuckerberg noted the Internet Research Agency (IRA) generated around 80,000 posts seen by approximately 126 million people and spent $100,000 on ads seen by 11 million. He announced plans to require advertiser authorization for political ads, verify large page managers, and support the Honest Ads Act. Zuckerberg emphasized that protecting the community and social mission takes priority over profits, noting Facebook serves more than 2 billion people. Chairman Grassley questioned the lack of transparency in data usage policies and the specific instances of improper data transfers beyond Kogan.
```

**Condensed account 1 of 3**, from segment summaries 1 to 23 (253 tokens)

```text
On April 10, 2018, a joint Senate hearing addressed Facebook's data privacy, focusing on the Cambridge Analytica scandal where approximately 87 million users' data was scraped. Senators John Thune, Chuck Grassley, Bill Nelson, Dianne Feinstein, and others questioned Facebook's failure to notify victims and its opaque consent models. Mark Zuckerberg testified, apologizing for the scandal and acknowledging that while he made changes in 2014, he failed to prevent harm from fake news and foreign interference. He confirmed that the Internet Research Agency spent $100,000 on ads seen by 11 million people and that around 200 apps were suspended pending audit. Zuckerberg stated Facebook serves over 2 billion users with $40 billion in 2017 revenue and emphasized that protecting the community takes priority over profits. He denied selling data to advertisers but acknowledged allowing targeted placement. Senators also discussed the Honest Ads Act, GDPR-style controls, and the need for federal standards. While Zuckerberg supported a 72-hour breach notification rule, he refused to disclose his hotel name or messenger contacts. The hearing concluded with a recess, leaving the committee to consider the need for stronger legislation versus self-regulation.
```

**Final summary** (359 tokens)

```text
On April 10, 2018, a joint Senate hearing addressed Facebook's data privacy, centering on the Cambridge Analytica scandal where approximately 87 million users' data was scraped. Mark Zuckerberg testified, apologizing for the failure to notify victims and acknowledging that while changes were made in 2014, he failed to prevent harm from fake news and foreign interference. He confirmed the Internet Research Agency spent $100,000 on ads seen by 11 million people and that around 200 apps were suspended pending audit. Zuckerberg stated Facebook serves over 2 billion users with $40 billion in 2017 revenue, emphasizing community protection over profits. He denied selling data to advertisers but acknowledged allowing targeted placement.

The hearing also covered content moderation and political neutrality. Zuckerberg detailed data collection practices and explained that content visibility adheres to Community Standards focused on Safety, Voice, and Equity, explicitly excluding political party affiliation from hate speech definitions. Regarding bias, Facebook reported that an investigation into the "Trending Topics" feature found no systematic bias, with approval rates for conservative and liberal topics being "virtually identical." On election integrity, Zuckerberg noted tracking 40+ elections, removing 70 Facebook and 65 Instagram accounts controlled by the Russia-based Internet Research Agency, and finding no evidence of Russian involvement in the "Release the Nunes Memo" campaign as of February 7, 2018.

Senators questioned the need for federal standards, including the Honest Ads Act and GDPR-style controls, while Zuckerberg supported a 72-hour breach notification rule but refused to disclose his hotel name or messenger contacts. The hearing concluded with a recess, leaving the committee to consider stronger legislation versus self-regulation.
```

**C-SPAN abstract**, program 500690: Facebook CEO Mark Zuckerberg Hearing on Data Privacy and Protection

```text
Facebook CEO Mark Zuckerberg testified before two Senate committees on the social media platform's use and protection of user data. The hearing was one of several held in response to revelations that the political consulting firm Cambridge Analytica improperly harvested data of up to 87 million Facebook users. Mr. Zuckerberg in his opening statement said his company was working on tools to prevent fake news, foreign interference, and hate speech. During his testimony, he apologized for Facebook's failures and admitted that not enough had been done to protect users' privacy, but he said his company has handled user data security responsibly.
```

## Step 10. Write the readable copies

The same files the August export wrote, in the same format, so the proposal's links keep working. They are the source text, the turn records, the three prompts, the chunks in pages of ten, every recorded output, and a copy of the call log.

In [10]:
(INPUTS_FOLDER / "source.txt").write_text(source_text, encoding="utf-8")
(INPUTS_FOLDER / "parsed_turns.json").write_text(json.dumps({
    "packageId": PACKAGE, "title": hearing["title"], "n_paragraphs": hearing["n_paragraphs"],
    "n_paragraphs_dropped": hearing["n_paragraphs_dropped"], "paragraph_rule": hearing["paragraph_rule"], "turns": turns,
}, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(LOGS_FOLDER / "call_log.jsonl").write_bytes((RUN_FOLDER / "status" / f"{PACKAGE}.jsonl").read_bytes())


def chunk_page(index):
    first = (index - 1) // 10 * 10 + 1
    return f"chunks_{first:02d}-{min(first + 9, len(chunks)):02d}.md"


lines = [f"# Model-input chunks: {PACKAGE}\n\n",
         "Reconstructed with the production tokenizer and 6,144-token budget. "
         f"All {len(chunks)} map user-prompt hashes match the saved run. "
         "The full inputs are divided into pages of up to ten chunks for reading. "
         "Text inside each fence is the exact chunk; headings and metadata are display labels.\n\n",
         "| Chunk | Tokens | Turn records represented | Split-turn flag |\n|---|---:|---:|---|\n"]
for i, c in enumerate(chunks, 1):
    lines.append(f"| [{i:02d}]({chunk_page(i)}#chunk-{i:02d}) | {c['n_tokens']:,} | {c['n_turns']} | {c['split_turn']} |\n")
(CHUNKS_FOLDER / "chunks.md").write_text("".join(lines), encoding="utf-8")
for start in range(0, len(chunks), 10):
    page = [f"# Model-input chunks {start + 1:02d}–{min(start + 10, len(chunks)):02d}: {PACKAGE}\n\n",
            "[All chunks and token counts](chunks.md). Text inside each fence is the verified model input; speaker metadata is a display label.\n"]
    for i, c in enumerate(chunks[start:start + 10], start + 1):
        page.append(f"\n## Chunk {i:02d}\n\nTokens: {c['n_tokens']:,}. Speaker metadata: {', '.join(c['speakers'])}. Split-turn flag: {c['split_turn']}.\n\n```text\n{c['text']}\n```\n")
    (CHUNKS_FOLDER / chunk_page(start + 1)).write_text("".join(page), encoding="utf-8")

lines = [f"# Recorded outputs: {PACKAGE}\n\n"
         "These are the output strings stored in the production log, without editorial corrections or new model generation. "
         "They include any sentence trimming performed by the original pipeline.\n\n"]
for c in calls:
    name = "Final summary" if c["stage"] == "reduce" else f"{c['stage'].title()} {c['index'] + 1:02d}"
    lines.append(f"## {name}\n\nStage `{c['stage']}`, level {c['level']}, {c['n_inputs']} input(s), {c['output_tokens']} output tokens.\n\n```text\n{c['output']}\n```\n\n")
lines.append(f"## C-SPAN reference\n\nSaved program ID: `{reference['cspan_id']}`. Title: **{reference['cspan_title']}**.\n\n```text\n{reference['abstract']}\n```\n")
(OUTPUTS_FOLDER / "outputs.md").write_text("".join(lines), encoding="utf-8")

lines = [f"# Prompts of the production run: {PROMPT_VERSION}\n\n",
         f"The three prompts below carry the version `{PROMPT_VERSION}` stamped on every logged call. "
         "Each call sends the system message and then the user message, with `{text}` or the list of summaries filled in. "
         f"Generation settings: {GENERATION}; the reduce call is capped at {REDUCE_MAX_TOKENS} tokens and asks for {REDUCE_TARGET_WORDS} words.\n\n"]
for name, system, user in (("Map prompt", MAP_SYSTEM, MAP_USER), ("Collapse prompt", COLLAPSE_SYSTEM, COLLAPSE_USER), ("Reduce prompt", REDUCE_SYSTEM, REDUCE_USER)):
    lines.append(f"## {name}\n\nSystem message:\n\n```text\n{system}\n```\n\nUser message:\n\n```text\n{user}\n```\n\n")
(INPUTS_FOLDER / "prompts.md").write_text("".join(lines), encoding="utf-8")

written = sorted(p.relative_to(EXAMPLE_FOLDER).as_posix() for p in EXAMPLE_FOLDER.rglob("*") if p.is_file() and p.suffix in (".txt", ".json", ".md", ".jsonl") and p.name != "README.md")
print(f"wrote {len(written)} files under {EXAMPLE_FOLDER.name}:")
for name in written:
    print("  ", name)

wrote 12 files under T1.4 - Running Example [Facebook-Cambridge Analytica Scandal Hearing]:
   01 - Inputs/Chunks/chunks.md
   01 - Inputs/Chunks/chunks_01-10.md
   01 - Inputs/Chunks/chunks_11-20.md
   01 - Inputs/Chunks/chunks_21-30.md
   01 - Inputs/Chunks/chunks_31-40.md
   01 - Inputs/Chunks/chunks_41-50.md
   01 - Inputs/Chunks/chunks_51-55.md
   01 - Inputs/parsed_turns.json
   01 - Inputs/prompts.md
   01 - Inputs/source.txt
   02 - Outputs/outputs.md
   03 - Logs and Verification/call_log.jsonl


## Step 11. Check the example

- 903 turns, 55 chunks within the budget, 59 calls verified in Step 8.
- The final summary in the log equals the stored summary and is within the 512-token cap.
- Every written chunk page lists its chunks, and `outputs.md` holds one section per call plus the abstract.

In [11]:
assert len(turns) == 903 and len(chunks) == 55 and len(calls) == 59
print("1) 903 turns, 55 chunks, 59 calls")

assert summary["summary_tokens"] <= REDUCE_MAX_TOKENS and summary["status"] == "ok"
print(f"2) the stored summary has {summary['summary_tokens']} tokens, within the {REDUCE_MAX_TOKENS}-token cap")

listed = sum(open(CHUNKS_FOLDER / f, encoding="utf-8").read().count("\n## Chunk ") for f in os.listdir(CHUNKS_FOLDER) if f.startswith("chunks_"))
sections = (OUTPUTS_FOLDER / "outputs.md").read_text(encoding="utf-8").count("\n## ")
assert listed == len(chunks) and sections == len(calls) + 1, f"{listed} chunks on the pages, {sections} output sections"
print(f"3) {listed} chunks on the pages; {sections} sections in outputs.md ({len(calls)} calls and the abstract)")

1) 903 turns, 55 chunks, 59 calls
2) the stored summary has 359 tokens, within the 512-token cap
3) 55 chunks on the pages; 60 sections in outputs.md (59 calls and the abstract)
